[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geo-di-lab/emerge-geoai-es/blob/main/docs/activities/intro_globe_analysis.ipynb)

# Actividad de EMERGE: análisis y mapeo de datos de GLOBE

Esta lección muestra cómo investigar datos de mosquitos y cobertura terrestre de GLOBE, calcular estadísticas y crear gráficos y mapas a escala mundial, estatal y de condado.

## Introducción a las herramientas

* **Python** es un lenguaje de programación popular que se utiliza para analizar datos, automatizar tareas, crear software, entrenar modelos de aprendizaje automático y mucho más. Lo utilizaremos para analizar datos de GLOBE Mosquito Habitat Mapper y GLOBE Land Cover.
* **Google Colab** es un entorno gratuito de programación en la nube que permite ejecutar Python desde el navegador. *¡No es necesario instalar ni descargar nada!*
* **Google Earth Engine** es una plataforma en la nube que incluye imágenes satelitales y otros conjuntos de datos geoespaciales. La utilizaremos para cargar datos ambientales.

**Para ejecutar el código:**

> Asegúrate de tener este cuaderno abierto en Google Colab. Desde el libro digital, haz clic en [![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geo-di-lab/emerge-geoai-es/blob/main/docs/activities/intro_globe_analysis.ipynb).

Cada bloque de código se denomina celda. Para ejecutar una celda, pasa el cursor sobre ella y haz clic en la flecha de la esquina superior izquierda, o haz clic dentro de la celda y presiona `Shift + Enter`.

Nota: La primera vez que ejecutes un bloque de código, Google Colab mostrará el mensaje `Warning: This notebook was not authored by Google`. Haz clic en `Run Anyway`.

In [ ]:
# Importar las bibliotecas
import pandas as pd                           # Trabajar con datos
pd.set_option("display.max_columns", None)    # Mostrar todas las columnas
import geopandas as gpd                       # Trabajar con datos espaciales
import numpy as np                            # Trabajar con números
from datetime import date                     # Trabajar con fechas
import matplotlib.pyplot as plt               # Crear gráficos
from matplotlib.colors import to_rgb          # Obtener colores
import branca.colormap as cm                  # Crear escalas de colores
import seaborn as sns                         # Usar paletas y crear gráficos
import folium                                 # Crear mapas interactivos

En el código anterior importamos bibliotecas de Python, que amplían las posibilidades de nuestro código. Cada biblioteca incluye muchas funciones diseñadas para realizar tareas específicas, como cargar un conjunto de datos, hacer cálculos, crear un gráfico y mucho más.

In [ ]:
# Este es un comentario, identificado por el símbolo # al inicio
# Los comentarios explican el código, pero no afectan su ejecución

# Mostrar la fecha de hoy
today = date.today()
print(f"La fecha de hoy es {today}.")

Una gran ventaja de Google Colab es que puedes escribir código en Python y ver el resultado directamente en el navegador.

## Mosquitos alrededor del mundo con GLOBE

Cargaremos los datos directamente desde el enlace. Todo permanecerá dentro de Google Colab en el navegador y no se descargará nada en tu computadora. Esta es una versión procesada de los datos, con algunas columnas renombradas y ciertos errores corregidos.

Los datos provienen de GLOBE Observer y se obtuvieron mediante [la API](https://www.globe.gov/globe-data/globe-api). Puedes consultar todos los pasos utilizados para procesar y limpiar los datos en el [capítulo 1 del currículo de EMERGE](https://geo-di-lab.github.io/emerge-lessons/docs/ch1/lesson6.html).

In [ ]:
mosquito = gpd.read_file('https://github.com/geo-di-lab/emerge-lessons/raw/refs/heads/main/docs/data/globe_mosquito.zip')
mosquito.head()

Consulta la lista de columnas:

In [ ]:
mosquito.info()

¿Cuántas filas contiene el conjunto de datos?

In [ ]:
len(mosquito)

Hubo 43,012 contribuciones de ciencia participativa entre 2018 y 2024. Ahora veremos en cuántos países se enviaron datos.

In [ ]:
len(mosquito['CountryCode'].unique())

Veamos los tipos de hábitats, es decir, fuentes de agua, que registraron las personas participantes.

In [ ]:
# Tipos generales de fuentes de agua
mosquito["WaterSourceType"].value_counts()

Estos son los tipos generales de fuentes de agua que las personas participantes reportaron a NASA GLOBE. La mayoría de los datos recopilados parecen corresponder a recipientes artificiales. Veamos algunos de los tipos más específicos incluidos en la otra columna:

In [ ]:
# Tipos más específicos de fuentes de agua
mosquito["WaterSource"].value_counts()

Crearemos un gráfico circular con los tipos más generales de la columna `WaterSourceType`.

In [ ]:
# Algunas opciones de paletas de colores
display(sns.color_palette(palette="rainbow"))
display(sns.color_palette(palette="CMRmap"))
display(sns.color_palette(palette="BrBG"))
display(sns.color_palette(palette="cubehelix"))
display(sns.color_palette(palette="Set2"))
display(sns.color_palette(palette="Set3"))
display(sns.color_palette(palette="tab20"))

In [ ]:
# Gráfico circular de los tipos de fuentes de agua
types = (
    mosquito[["SiteId", "WaterSourceType"]]
    .groupby("WaterSourceType", as_index=False)
    .count()
)

# Crear el gráfico circular
plt.figure(figsize=(5, 5))
patches, texts = plt.pie(
    colors=sns.color_palette("Set2"),  # Nombre de la paleta elegida
    x=types["SiteId"]
)
plt.title(
    "Observaciones de mosquitos de GLOBE: "
    "tipos generales de fuentes de agua"
)
plt.legend(
    patches,
    types["WaterSourceType"],
    loc="center left",
    bbox_to_anchor=(1, 0.5),
    frameon=False
)
plt.show()

¿Cuál es el promedio de larvas registradas por país?

In [ ]:
mosquito_avg = mosquito.groupby('CountryCode')['LarvaeCountProcessed'].mean()
mosquito_avg

Crearemos un mapa que muestre el promedio de larvas por país. Los [límites generalizados de los países](https://hub.arcgis.com/datasets/esri::world-countries-generalized/about) provienen de Esri, Garmin y la Agencia Central de Inteligencia de Estados Unidos, mediante *The World Factbook*. Los límites están generalizados para que el procesamiento de los datos y la carga de las visualizaciones sean más rápidos.

Los códigos ISO alfa-3 provienen de la capa [World Countries](https://hub.arcgis.com/datasets/esri::world-countries/about), elaborada con datos de Esri, Garmin, la Agencia Central de Inteligencia de Estados Unidos y la Organización Internacional de Normalización (ISO).

In [ ]:
# Cargar los límites de los países
countries = gpd.read_file(
    "https://github.com/geo-di-lab/emerge-lessons/"
    "raw/refs/heads/main/docs/data/"
    "world_countries_general.geojson"
).to_crs(epsg=4326)

# Agregar los datos de mosquitos correspondientes a cada país
mosquito_avg = countries.merge(
    mosquito_avg,
    left_on="iso3",
    right_on="CountryCode",
    how="left"
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

# Crear un mapa del promedio de larvas por país
mosquito_avg.plot(
    column="LarvaeCountProcessed",
    cmap="viridis",
    legend=True,
    vmin=0,
    vmax=50,
    ax=ax,
    missing_kwds={"color": "lightgrey"}
)
plt.title(
    "Observaciones de mosquitos de GLOBE: "
    "promedio de larvas"
)
ax.axis("off")
plt.show()

Ahora crearemos un mapa *interactivo* que muestre el total de observaciones de GLOBE por país.

In [ ]:
# Obtener el total de observaciones de GLOBE por país
mosquito_obs = (
    mosquito.groupby("CountryCode")
    .size()
    .reset_index(name="GLOBE_Observations")
)
mosquito_obs = countries.merge(
    mosquito_obs,
    left_on="iso3",
    right_on="CountryCode",
    how="left"
).dropna(subset=["GLOBE_Observations"])

In [ ]:
# Cargar una escala de colores que comienza en 1 y termina en 100
# El azul representa 100 observaciones o más; el verde y el amarillo,
# valores más cercanos a una observación
colors = cm.linear.YlGnBu_03.scale(1, 100)
colors

In [ ]:
map = folium.Map(
    location=[0, 0],
    zoom_start=3,
    tiles="CartoDB positron"
)

# Crear un mapa interactivo de las observaciones de GLOBE por país
folium.GeoJson(
    geo_data=mosquito_obs.to_json(),
    data=mosquito_obs,
    key_on="feature.properties.name",
    tooltip=folium.features.GeoJsonTooltip(
        fields=["name", "GLOBE_Observations"],
        aliases=["País:", "Observaciones:"]
    ),
    style_function=lambda feature: {
        "fillColor": colors(
            feature["properties"]["GLOBE_Observations"]
        ),
        "fillOpacity": 0.8,
        "color": "grey",
        "weight": 1
    }
).add_to(map)

display(map)

## Cobertura terrestre alrededor del mundo con GLOBE

Al igual que con los datos de mosquitos, cargaremos directamente desde el enlace los datos de cobertura terrestre de GLOBE.

In [ ]:
land_cover = gpd.read_file('https://github.com/geo-di-lab/emerge-lessons/raw/refs/heads/main/docs/data/globe_land_cover.zip')
land_cover.head()

In [ ]:
len(land_cover)

Una parte útil del conjunto de datos de cobertura terrestre son las clasificaciones MUC. MUC, o Clasificación Modificada de la UNESCO, es un sistema que incluye distintos tipos de uso del suelo y nos ayuda a comprender los hábitats de todo el mundo.

¿Cuáles son los códigos MUC más comunes en cada país?

In [ ]:
# Encontrar el MUC más común de cada país
muc = (
    land_cover.groupby("CountryCode")["MucDescription"]
    .apply(
        lambda x: (
            x.value_counts().idxmax()
            if not x.value_counts().empty
            else None
        )
    )
    .reset_index(name="MucDescription")
)

# Agregar la cantidad de observaciones con ese código MUC
muc["Count"] = (
    land_cover.groupby("CountryCode")["MucDescription"]
    .apply(lambda x: x.value_counts().max())
    .values
)

# Agregar el total de observaciones de GLOBE
muc["GLOBE_Observations"] = (
    land_cover.groupby("CountryCode").size().values
)

muc

In [ ]:
# Agregar los datos a los límites de los países
muc = countries.merge(
    muc,
    left_on="iso3",
    right_on="CountryCode",
    how="left"
)

# Crear categorías generales
# Estos valores permanecen en inglés porque deben coincidir con
# el contenido original de la columna MucDescription
muc_list = [
    "Barren",
    "Closed Forest",
    "Cultivated",
    "Herbaceous",
    "Open Water",
    "Trees",
    "Urban",
    "Wetlands",
    "Woodland"
]

# Simplificar las descripciones mediante las categorías anteriores
for muc_code in muc_list:
    muc.loc[
        muc["MucDescription"].str.contains(muc_code, na=False),
        "MucDescriptionShort"
    ] = muc_code

In [ ]:
fig, ax = plt.subplots(figsize=(11, 5))

# Crear un mapa de los códigos MUC más comunes por país
muc.plot(
    column="MucDescriptionShort",
    cmap="viridis",
    legend=True,
    ax=ax,
    missing_kwds={
        "color": "lightgrey",
        "label": "Sin datos"
    },
    legend_kwds={
        "loc": "lower left",
        "frameon": False
    }
)
plt.title(
    "Cobertura terrestre de GLOBE: "
    "códigos MUC más comunes"
)
plt.show()

Observa las zonas grises. No se registraron observaciones de GLOBE en esos países, por lo que los eliminaremos del conjunto de datos para facilitar la visualización.

In [ ]:
muc = muc.dropna(subset=['GLOBE_Observations'])

### Desafío: crear un mapa interactivo de la cobertura terrestre

Al igual que en el mapa de observaciones de mosquitos, crearemos un mapa con la cantidad de observaciones de cobertura terrestre de GLOBE. Además del nombre del país y la cantidad de observaciones de GLOBE, agrega el código MUC más común, `MucDescriptionShort`, al cuadro de información que aparece al pasar el cursor sobre cada país.

Utiliza las siguientes columnas:

- `name` para el nombre del país
- `MucDescriptionShort` para el código MUC más común
- `GLOBE_Observations` para la cantidad de observaciones de GLOBE registradas y para determinar el color de cada país

En resumen, el color de cada país debe basarse en `GLOBE_Observations`. El cuadro emergente debe incluir el nombre del país, la cantidad de observaciones de GLOBE y el código MUC más común.

Puedes consultar como referencia el código del mapa interactivo de observaciones de mosquitos, que incluimos a continuación. La estructura técnica permanece en inglés porque utiliza los nombres de las columnas del conjunto de datos. Si necesitas ayuda, abre la sección de respuesta para ver una manera de crear el mapa.

```python
map = folium.Map(location=[0, 0], zoom_start=3, tiles="CartoDB positron")

# Crear un mapa interactivo de las observaciones de GLOBE por país
folium.GeoJson(
    geo_data=mosquito_obs.to_json(),
    data=mosquito_obs,
    key_on="feature.properties.name",
    tooltip=folium.features.GeoJsonTooltip(
        fields=["name", "GLOBE_Observations"],
        aliases=["País:", "Observaciones:"]
    ),
    style_function=lambda feature: {
        "fillColor": colors(
            feature["properties"]["GLOBE_Observations"]
        ),
        "fillOpacity": 0.8,
        "color": "grey",
        "weight": 1
    }
).add_to(map)

display(map)
```

In [ ]:
map = folium.Map(
    location=[0, 0],
    zoom_start=3,
    tiles="CartoDB positron"
)

# Completa estas cuatro variables
geo_data = None
map_data = None
tooltip_fields = []
tooltip_aliases = []

if (
    geo_data is None
    or map_data is None
    or not tooltip_fields
    or not tooltip_aliases
):
    print(
        "Completa geo_data, map_data, tooltip_fields "
        "y tooltip_aliases antes de crear el mapa."
    )
else:
    folium.GeoJson(
        geo_data=geo_data,
        data=map_data,
        key_on="feature.properties.name",
        tooltip=folium.features.GeoJsonTooltip(
            fields=tooltip_fields,
            aliases=tooltip_aliases
        ),
        style_function=lambda feature: {
            "fillColor": colors(
                feature["properties"]["GLOBE_Observations"]
            ),
            "fillOpacity": 0.8,
            "color": "grey",
            "weight": 1
        }
    ).add_to(map)

    display(map)

In [ ]:
#@title Abrir para ver la respuesta

map = folium.Map(
    location=[0, 0],
    zoom_start=3,
    tiles="CartoDB positron"
)

folium.GeoJson(
    geo_data=muc.to_json(),
    data=muc,
    key_on="feature.properties.name",
    tooltip=folium.features.GeoJsonTooltip(
        fields=[
            "name",
            "GLOBE_Observations",
            "MucDescriptionShort"
        ],
        aliases=[
            "País:",
            "Observaciones:",
            "MUC más común:"
        ]
    ),
    style_function=lambda feature: {
        "fillColor": colors(
            feature["properties"]["GLOBE_Observations"]
        ),
        "fillOpacity": 0.8,
        "color": "grey",
        "weight": 1
    }
).add_to(map)

display(map)

## Mosquitos en tu estado

Ahora veremos los datos de GLOBE correspondientes a tu estado y, más adelante, a tu condado. Escribe a continuación el nombre de tu estado **en inglés**, tal como aparece en los datos del Censo de Estados Unidos.

Los archivos de límites provienen de la [Oficina del Censo de Estados Unidos](https://www.census.gov/geographies/mapping-files/time-series/geo/cartographic-boundary.html). Utilizaremos los mismos datos de mosquitos de la primera parte del cuaderno. Estos datos provienen de GLOBE Observer, se obtuvieron mediante [la API](https://www.globe.gov/globe-data/globe-api) y se procesaron en el [capítulo 1 de nuestro libro digital](https://geo-di-lab.github.io/emerge-lessons/docs/ch1/lesson6.html).

In [ ]:
# Escribe el nombre del estado en inglés
state_name = "Your State Name"

# Por ejemplo:
# state_name = "Florida" 

In [ ]:
# Cargar los límites de los condados
counties = gpd.read_file(
    "https://github.com/geo-di-lab/emerge-lessons/"
    "raw/refs/heads/main/docs/data/us_counties.zip"
).to_crs("EPSG:4326")

# Filtrar todos los condados del estado seleccionado
state_counties = counties.loc[
    counties["STATE_NAME"] == state_name
]

# Obtener el límite estatal mediante la unión de los condados
state = state_counties[["geometry"]].dissolve()

state.plot()

In [ ]:
# Crear un mapa vacío centrado en el estado
map = folium.Map(tiles="Cartodb dark_matter")

# Obtener todos los datos de mosquitos de GLOBE dentro del estado
mosquito_state = (
    gpd.sjoin(
        mosquito,
        state,
        how="inner",
        predicate="intersects"
    )
    .drop(columns=["index_right"])
    .reset_index(drop=True)
)

# Agregar los límites de los condados al mapa
folium.GeoJson(
    state_counties.to_json(),
    name="Límites de los condados",
    tooltip=folium.features.GeoJsonTooltip(
        fields=["NAMELSAD"],
        aliases=["Condado:"]
    ),
    style_function=lambda feature: {
        "fillColor": "transparent",
        "color": "grey",
        "weight": 1
    }
).add_to(map)

# Agregar cada observación como un círculo verde
for idx, row in mosquito_state.iterrows():
    popup_content = (
        f"<b>Fecha:</b> {row['MeasuredDate']}<br>"
        f"<b>Fuente de agua:</b> {row['WaterSourceType']}<br>"
        f"<b>Longitud:</b> {row['MeasurementLongitude']}<br>"
        f"<b>Latitud:</b> {row['MeasurementLatitude']}"
    )

    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        popup=folium.Popup(
            popup_content,
            max_width=300
        ),
        radius=6,
        color="black",
        weight=1,
        fillColor="lightgreen",
        fillOpacity=0.5
    ).add_to(map)

# Agregar una opción para ocultar los límites
folium.LayerControl().add_to(map)

# Acercar el mapa al estado
minx, miny, maxx, maxy = state.bounds.values[0]
bounds = [[miny, minx], [maxy, maxx]]
map.fit_bounds(bounds)

# Mostrar el mapa
display(map)

Comprueba si hay puntos dentro de tu condado. Si no los hay, busca un condado cercano que tenga puntos. Escribe a continuación el nombre del condado elegido **en inglés**:

In [ ]:
# Escribe el nombre del condado en inglés
county_name = "Your County Name"

# El nombre debe terminar con la palabra "County"

# Por ejemplo:
# county_name = "Broward County" 

## Mosquitos en tu condado

Obtén el contorno específico de tu condado:

In [ ]:
county = state_counties.loc[
    state_counties["NAMELSAD"] == county_name
]
county.plot()

Ahora utilizaremos esos límites para filtrar todos los puntos de GLOBE ubicados dentro de tu condado entre 2018 y 2024.

In [ ]:
data_county = mosquito.sjoin(
    county,
    how="inner",
    predicate="within"
)

num_total = len(data_county)

print(
    f"Se registraron {num_total} puntos de GLOBE dentro de "
    f"{county_name} entre 2018 y 2024."
)

num_eliminated = len(
    data_county[
        data_county["BreedingGroundEliminated"] == "true"
    ]
)

if num_total > 0:
    percentage = round(num_eliminated * 100 / num_total)
    print(
        f"De esos puntos, las personas participantes mitigaron "
        f"correctamente {num_eliminated} ({percentage} %). "
        "Esto reduce el riesgo de que los mosquitos ocupen ese "
        "lugar en el futuro."
    )
else:
    print(
        "No se encontraron puntos de GLOBE en el condado "
        "seleccionado. Elige un condado cercano con observaciones."
    )

Si no hay puntos de GLOBE en tu condado, elige un condado cercano que tenga al menos uno. Puedes identificarlo en el mapa estatal anterior.

In [ ]:
data_county.head()

¿Cuántas observaciones de mosquitos se enviaron cada año a GLOBE Observer desde tu condado? Crearemos un gráfico de barras para averiguarlo.

In [ ]:
# Agregar una columna con el año
data_county["MeasuredYear"] = (
    data_county["MeasuredAt"].dt.year
)

# Crear un gráfico de barras de las observaciones por año
years = (
    data_county[["SiteId", "MeasuredYear"]]
    .groupby("MeasuredYear", as_index=False)
    .count()
)
plt.bar(years["MeasuredYear"], years["SiteId"])
plt.title(
    "Observaciones de mosquitos por año",
    loc="left"
)
plt.title(county_name, loc="right")
plt.show()

Crearemos gráficos circulares de los tipos de fuentes de agua, tanto generales como específicos, donde se reportaron mosquitos dentro de este condado.

In [ ]:
# Contar cada tipo general de fuente de agua
types = (
    data_county[["SiteId", "WaterSourceType"]]
    .groupby("WaterSourceType", as_index=False)
    .count()
)

# Crear el gráfico circular
plt.figure(figsize=(5, 5))
patches, texts = plt.pie(
    x=types["SiteId"],
    colors=sns.color_palette("Set2")
)
plt.title(
    f"Observaciones de mosquitos de GLOBE en {county_name}: "
    "tipos generales de fuentes de agua"
)
plt.legend(
    patches,
    types["WaterSourceType"],
    loc="center left",
    bbox_to_anchor=(1, 0.5),
    frameon=False
)
plt.show()

In [ ]:
# Contar cada tipo específico de fuente de agua
types = (
    data_county[["SiteId", "WaterSource"]]
    .groupby("WaterSource", as_index=False)
    .count()
)

# Crear el gráfico circular
plt.figure(figsize=(5, 5))
patches, texts = plt.pie(
    x=types["SiteId"],
    colors=sns.color_palette("Set2")
)
plt.title(
    f"Observaciones de mosquitos de GLOBE en {county_name}: "
    "tipos específicos de fuentes de agua"
)
plt.legend(
    patches,
    types["WaterSource"],
    loc="center left",
    bbox_to_anchor=(1, 0.5),
    frameon=False
)
plt.show()

## Datos de teledetección para tu condado

Si todavía no has utilizado Google Earth Engine, sigue [este tutorial](https://geo-di-lab.github.io/emerge-lessons/docs/ch1/lesson5a.html) para crear una cuenta y un proyecto de Earth Engine.

In [ ]:
# Escribe a continuación el ID de tu proyecto de Google Earth Engine
projectID = "Write Your Project ID"

# Por ejemplo:
# projectID = "emerge-lessons-23148" 

In [ ]:
import ee
import geemap
import geopandas as gpd

ee.Authenticate()

# Inicializar Earth Engine con el ID del proyecto indicado arriba
ee.Initialize(project=projectID)

In [ ]:
# Convertir los límites del condado al formato de Earth Engine
region = geemap.geopandas_to_ee(county)

# Crear un mapa vacío centrado en el condado
county_center = county.iloc[0].geometry.centroid

map = geemap.Map(
    center=[county_center.y, county_center.x],
    zoom=10
)
map.add_basemap("CartoDB.DarkMatter")

Ahora representaremos en un mapa la temperatura de la superficie terrestre de todo el condado. Primero, define el intervalo de fechas que se utilizará para calcular la temperatura promedio.

In [ ]:
start_date = "2025-06-01"
end_date = "2025-07-01" 

Visualizaremos datos de teledetección de NASA disponibles mediante Google Earth Engine: [MOD11A1.061 Terra Land Surface Temperature and Emissivity Daily Global 1km](https://developers.google.com/earth-engine/datasets/catalog/MODIS_061_MOD11A1).

Los valores de temperatura de la superficie terrestre están escalados. Para recuperar la temperatura original en kelvin, el valor debe multiplicarse por 0.02. Después podemos convertirlo a grados Fahrenheit.

In [ ]:
def to_fahrenheit(lst):
    """Convierte un valor escalado de MODIS LST a grados Fahrenheit."""
    celsius = lst * 0.02 - 273.15
    fahrenheit = celsius * 1.8 + 32
    return fahrenheit


def to_lst(fahrenheit):
    """Convierte grados Fahrenheit al valor escalado de MODIS LST."""
    celsius = (fahrenheit - 32) / 1.8
    lst = (celsius + 273.15) / 0.02
    return lst

In [ ]:
# Cargar los datos de Google Earth Engine
lst = (
    ee.ImageCollection("MODIS/061/MOD11A1")
    .filterDate(start_date, end_date)
    .select("LST_Day_1km")
    .median()
    .clip(region)
)

# Definir la escala de colores
colors = [
    "040274", "040281", "0502a3", "0502b8", "0502ce", "0502e6",
    "0602ff", "235cb1", "307ef3", "269db1", "30c8e2", "32d3ef",
    "3be285", "3ff38f", "86e26f", "3ae237", "b5e22e", "d6e21f",
    "fff705", "ffd611", "ffb613", "ff8b13", "ff6e08", "ff500d",
    "ff0000", "de0101", "c21301", "a71001", "911003"
]

# Definir los valores mínimo y máximo y la escala de colores
lst_vis = {
    "min": to_lst(50),
    "max": to_lst(100),
    "palette": colors
}

# Agregar los datos al mapa
map.addLayer(
    lst,
    lst_vis,
    "Temperatura de la superficie terrestre"
)

display(map)

La siguiente barra de colores muestra qué representan los colores del mapa: la temperatura de la superficie terrestre en grados Fahrenheit.

In [ ]:
plt.figure(figsize=(len(colors), 1))
plt.imshow([[to_rgb(f"#{color}") for color in colors]])

plt.text(
    -0.6,
    0.1,
    "50 °F",
    va="center",
    ha="right",
    fontsize=24
)
plt.text(
    len(colors) - 0.4,
    0.1,
    "100 °F",
    va="center",
    ha="left",
    fontsize=24
)

plt.axis("off")
plt.show()

A continuación, calcularemos la temperatura promedio de todo el condado.

In [ ]:
mean_lst = (
    lst.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=region,
        scale=1000,
        maxPixels=1e12
    )
    .get("LST_Day_1km")
    .getInfo()
)

print(
    f"La temperatura promedio de {county_name} entre "
    f"{start_date} y {end_date} es "
    f"{round(to_fahrenheit(mean_lst), 2)} °F."
)

Ahora visualizaremos el uso y la cobertura del suelo del condado mediante el conjunto de datos [USFS Landscape Change Monitoring System v2024.10 (CONUS and OCONUS)](https://developers.google.com/earth-engine/datasets/catalog/USFS_GTAC_LCMS_v2024-10#bands).

In [ ]:
# Uso del suelo

# Reiniciar el mapa
map = geemap.Map(
    center=[county_center.y, county_center.x],
    zoom=10
)
map.add_basemap("CartoDB.DarkMatter")

# Definir la paleta de colores del uso del suelo
palette = [
    "fbff97",
    "e6558b",
    "004e2b",
    "9dbac5",
    "a6976a",
    "1b1716"
]
visual = {
    "min": 1,
    "max": 6,
    "palette": palette
}

# Agregar el uso del suelo de 1985
landcover_1985 = (
    ee.ImageCollection("USFS/GTAC/LCMS/v2024-10")
    .filterDate("1985", "1986")
    .filter('study_area == "CONUS"')
    .first()
    .clip(region)
)

map.addLayer(
    landcover_1985.select("Land_Use"),
    visual,
    "Uso del suelo en 1985"
)

# Agregar el uso del suelo de 2024
landcover_2024 = (
    ee.ImageCollection("USFS/GTAC/LCMS/v2024-10")
    .filterDate("2024", "2025")
    .filter('study_area == "CONUS"')
    .first()
    .clip(region)
)

map.addLayer(
    landcover_2024.select("Land_Use"),
    visual,
    "Uso del suelo en 2024"
)

display(map)

$$\color{#fbff97}Agricultura$$
$$\color{#e6558b}Desarrollo\ urbano$$
$$\color{#004e2b}Bosque$$
$$\color{#9dbac5}Otro$$
$$\color{#a6976a}Pastizal\ o\ terreno\ de\ pastoreo$$

Crearemos una función que calcule el porcentaje de cada tipo de uso y cobertura del suelo presente en el condado.

In [ ]:
def land_stats(image, name, labels):
    """
    Calcula el porcentaje de cada tipo de uso o cobertura del suelo
    dentro del condado.

    Argumentos:
        image: imagen de Earth Engine
        name: nombre de la banda, Land_Use o Land_Cover
        labels: diccionario de etiquetas
    """
    count = (
        image.select(name)
        .reduceRegion(
            reducer=ee.Reducer.frequencyHistogram(),
            geometry=region,
            scale=30,
            maxPixels=1e12
        )
        .getInfo()
    )

    land_cover_counts = {}

    for key, value in labels.items():
        if key in count.get(name, {}):
            land_cover_counts[value] = count[name][key]

    total_pixels = sum(land_cover_counts.values())

    if total_pixels == 0:
        print("No se encontraron píxeles válidos en la región.")
        return

    for label, num in land_cover_counts.items():
        percent = round(100 * num / total_pixels, 1)
        if percent > 0:
            print(f"{label}: {percent} %")

Obtén los porcentajes de uso del suelo correspondientes a 1985.

In [ ]:
land_use_labels = {
    "1": "Agricultura",
    "2": "Desarrollo urbano",
    "3": "Bosque",
    "4": "Otro",
    "5": "Pastizal o terreno de pastoreo",
    "6": "Máscara de área no procesada"
}

land_stats(
    landcover_1985,
    "Land_Use",
    land_use_labels
)

Ahora repetiremos el cálculo para 2024.

In [ ]:
land_stats(
    landcover_2024,
    "Land_Use",
    land_use_labels
)

Visualiza la cobertura terrestre en un mapa.

In [ ]:
# Cobertura terrestre

# Reiniciar el mapa
map = geemap.Map(
    center=[county_center.y, county_center.x],
    zoom=10
)
map.add_basemap("CartoDB.DarkMatter")

# Definir la paleta de colores
palette = [
    "004e2b", "009344", "61bb46", "acbb67", "8b8560",
    "cafd4b", "f89a1c", "8fa55f", "bebb8e", "e5e98a",
    "ddb925", "893f54", "e4f5fd", "00b6f0", "1b1716"
]
visual = {
    "min": 1,
    "max": 15,
    "palette": palette
}

# Agregar la cobertura terrestre de 1985 y 2024
map.addLayer(
    landcover_1985.select("Land_Cover"),
    visual,
    "Cobertura terrestre en 1985"
)
map.addLayer(
    landcover_2024.select("Land_Cover"),
    visual,
    "Cobertura terrestre en 2024"
)

display(map)

$$\color{#004e2b}Árboles$$
$$\color{#61bb46}Mezcla\ de\ arbustos\ y\ árboles$$
$$\color{#acbb67}Mezcla\ de\ pastos,\ hierbas\ y\ árboles$$
$$\color{#8b8560}Mezcla\ de\ terreno\ sin\ vegetación\ y\ árboles$$
$$\color{#f89a1c}Arbustos$$
$$\color{#8fa55f}Mezcla\ de\ pastos,\ hierbas\ y\ arbustos$$
$$\color{#bebb8e}Mezcla\ de\ terreno\ sin\ vegetación\ y\ arbustos$$
$$\color{#e5e98a}Pastos\ y\ hierbas$$
$$\color{#ddb925}Mezcla\ de\ terreno\ sin\ vegetación,\ pastos\ y\ hierbas$$
$$\color{#893f54}Terreno\ sin\ vegetación\ o\ impermeable$$
$$\color{#e4f5fd}Nieve\ o\ hielo$$

Calcula el porcentaje de cada tipo de cobertura terrestre.

In [ ]:
land_cover_labels = {
    "1": "Árboles",
    "2": "Mezcla de arbustos altos y árboles (solo Alaska)",
    "3": "Mezcla de arbustos y árboles",
    "4": "Mezcla de pastos, hierbas y árboles",
    "5": "Mezcla de terreno sin vegetación y árboles",
    "6": "Arbustos altos (solo Alaska)",
    "7": "Arbustos",
    "8": "Mezcla de pastos, hierbas y arbustos",
    "9": "Mezcla de terreno sin vegetación y arbustos",
    "10": "Pastos y hierbas",
    "11": "Mezcla de terreno sin vegetación, pastos y hierbas",
    "12": "Terreno sin vegetación o impermeable",
    "13": "Nieve o hielo",
    "14": "Agua",
    "15": "Máscara de área no procesada"
}

In [ ]:
land_stats(
    landcover_1985,
    "Land_Cover",
    land_cover_labels
)

In [ ]:
land_stats(
    landcover_2024,
    "Land_Cover",
    land_cover_labels
)

## Integración de los análisis: obtener datos de teledetección para un punto de GLOBE

Consulta la lista de observaciones del condado seleccionado.

In [ ]:
data_county

Elige una de estas observaciones y anota su índice, es decir, el número situado en la columna izquierda, como `6643` o `29024`. Escríbelo en la siguiente celda.

In [ ]:
# Sustituye None por el índice de la observación elegida
index = None

# Por ejemplo:
# index = 29024

Muestra los datos correspondientes al punto con ese índice:

In [ ]:
if index is None:
    raise ValueError(
        "Sustituye None por el índice de una observación "
        "antes de ejecutar esta celda."
    )

point = data_county[data_county.index == index]

if point.empty:
    raise ValueError(
        f"No se encontró una observación con el índice {index}."
    )

point

Obtendremos datos ambientales de temperatura de la superficie terrestre, precipitación y vegetación para el punto seleccionado. Puedes consultar más información sobre este código en el [capítulo 4 del libro digital](https://geo-di-lab.github.io/emerge-lessons/docs/ch4/lesson2.html).

In [ ]:
def mask_s2_clouds(image):
    """
    Enmascara las nubes de una imagen Sentinel-2 mediante la banda QA.

    Argumentos:
        image (ee.Image): imagen de Sentinel-2

    Devuelve:
        ee.Image: imagen de Sentinel-2 con las nubes enmascaradas
    """
    qa = image.select("QA60")

    # Los bits 10 y 11 representan nubes y cirros, respectivamente
    cloud_bit_mask = 1 << 10
    cirrus_bit_mask = 1 << 11

    # Ambas señales deben ser cero para indicar condiciones despejadas
    mask = (
        qa.bitwiseAnd(cloud_bit_mask)
        .eq(0)
        .And(
            qa.bitwiseAnd(cirrus_bit_mask).eq(0)
        )
    )

    return image.updateMask(mask).divide(10000)


def get_data_for_point(point):
    """Obtiene datos ambientales para una observación de GLOBE."""

    # Extraer la fecha de la observación
    point_date = ee.Date(point.get("MeasuredDate"))
    day_before = point_date.advance(-1, "day")
    month_before = point_date.advance(-30, "day")

    # Temperatura de la superficie terrestre de MODIS de NASA
    lst = (
        ee.ImageCollection("MODIS/061/MOD11A1")
        .filterDate(month_before, point_date)
        .select([
            "LST_Day_1km",
            "QC_Day",
            "LST_Night_1km",
            "QC_Night",
            "Clear_day_cov",
            "Clear_night_cov"
        ])
        .median()
        # Obtener los valores en la ubicación del punto
        .reduceRegion(
            reducer=ee.Reducer.median(),
            geometry=point.geometry(),
            scale=1000
        )
    )
    point = point.set(lst)

    lst_value = point.get("LST_Day_1km").getInfo()
    point = point.set(
        "LST_Day_1km_F",
        to_fahrenheit(lst_value)
    )

    # Mediana de Sentinel-2 durante el mes anterior
    # Se utilizará para calcular el NDVI
    sentinel2 = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterDate(month_before, point_date)
        .map(mask_s2_clouds)
        .median()
        # Obtener la mediana de las bandas en la ubicación
        .reduceRegion(
            reducer=ee.Reducer.median(),
            geometry=point.geometry(),
            scale=10
        )
    )
    point = point.set(sentinel2)

    # Precipitación de Daymet V4 de NASA
    rain = (
        ee.ImageCollection("NASA/ORNL/DAYMET_V4")
        .filterDate(day_before, point_date)
        .select("prcp")  # Precipitación diaria total
        .sum()
        # Obtener la mediana en la ubicación del punto
        .reduceRegion(
            reducer=ee.Reducer.median(),
            geometry=point.geometry(),
            scale=1000
        )
    )
    point = point.set(rain)

    # Agregar información adicional
    b8 = ee.Number(point.get("B8"))
    b4 = ee.Number(point.get("B4"))
    point = point.set(
        "NDVI",
        b8.subtract(b4).divide(b8.add(b4))
    )

    return point

Utiliza esta función para obtener los datos de precipitación, temperatura y vegetación del punto seleccionado.

In [ ]:
point_fc = geemap.geopandas_to_ee(point).first()
result = get_data_for_point(point_fc)

Muestra los datos:

El índice de vegetación de diferencia normalizada, o NDVI, es un valor entre -1 y 1 que representa la cantidad de vegetación. Los valores altos, cercanos a 1, indican zonas con vegetación abundante, como bosques. Los valores bajos, cercanos a -1, indican zonas con poca vegetación.

In [ ]:
print(
    f"Observación de mosquitos de GLOBE en "
    f"{county_name}, {state_name}"
)
print(
    f"Fecha: {result.get('MeasuredDate').getInfo()}"
)
print(
    f"Fuente de agua: {result.get('WaterSource').getInfo()}"
)
print(
    "Precipitación: "
    f"{round(result.get('prcp').getInfo(), 2)} mm"
)
print(
    "Temperatura de la superficie terrestre: "
    f"{round(result.get('LST_Day_1km_F').getInfo(), 2)} °F"
)
print(
    "Índice de vegetación de diferencia normalizada (NDVI): "
    f"{round(result.get('NDVI').getInfo(), 2)}"
)

A continuación encontrarás más recursos de GLOBE y EMERGE:

- [Sitio web de EMERGE](https://geoemerge.com/) y [currículo](https://geo-di-lab.github.io/emerge-lessons)
- [Recursos para educadores de GLOBE](https://www.globe.gov/do-globe/resources/teaching-resources)
- [Biblioteca de recursos de GLOBE Mosquito Habitat Mapper](https://observer.globe.gov/do-globe-observer/mosquito-habitats/resource-library)
- [Biblioteca de recursos de GLOBE Land Cover](https://observer.globe.gov/do-globe-observer/land-cover)